In [2]:
# Imports needed
import pandas as pd
import json
import sklearn
import nltk
import seaborn as sns

In [3]:
# Load only the first 10,000 rows from the dataset
review_data = pd.read_json('yelp_academic_dataset_review.json', lines=True, nrows=10000)

# Show columns to identify location fields
print("Columns in dataset:", review_data.columns.tolist())

# Display the first 10,000 rows
print(f"Loaded {len(review_data)} rows")
print("\nFirst 10,000 rows of the dataset:")
#review_data

# Load only the first # rows from the dataset
# Get all business data to make sure when merging, there are some guaranteed business ID matches from review data read in
business_data = pd.read_json('yelp_academic_dataset_business.json', lines=True)#, nrows=25)

# Show columns to identify location fields
print("Columns in dataset:", business_data.columns.tolist())

# Display the first # rows
print(f"Loaded {len(business_data)} rows")
print("\nFirst # rows of the dataset:")
#business_data

# Only keep businesses that appear in reviews
business_data = business_data[business_data['business_id'].isin(review_data['business_id'])]

Columns in dataset: ['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date']
Loaded 10000 rows

First 10,000 rows of the dataset:
Columns in dataset: ['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours']
Loaded 150346 rows

First # rows of the dataset:


## Cuisine Types

In [4]:
# Only include restaurants (remove doctors, shipping centers, etc.)
business_data = business_data[business_data['categories'].str.contains('Restaurants', na=False)]

def get_cuisine(categories):
    categories = str(categories)

    cuisines = [
        'Chinese', 
        'Italian', 
        'Mexican', 
        'French', 
        'Japanese', 
        'Korean', 
        'Mediterranean', 
        'Vietnamese', 
        'American'
    ]

    for cuisine in cuisines:
        if cuisine in categories:
            return cuisine
    return None

In [5]:
print("Review rows:", len(review_data))
print("Business rows:", len(business_data))
print(business_data['categories'].head(30))

# Merge two datasets (review, business)

business_data['cuisine_label'] = business_data['categories'].apply(get_cuisine)

merged_data = review_data.merge(business_data[['business_id', 'cuisine_label']], on='business_id')

print("After merge:", len(merged_data))
print("Null labels after merge:", merged_data['cuisine_label'].isna().sum())

# Drop rows with no label
merged_data = merged_data.dropna(subset=['cuisine_label'])
print("After filtering:", len(merged_data))

Review rows: 10000
Business rows: 2210
3      Restaurants, Food, Bubble Tea, Coffee & Tea, B...
14           Food, Delis, Italian, Bakeries, Restaurants
15                     Sushi Bars, Restaurants, Japanese
19                                   Korean, Restaurants
20     Coffee & Tea, Food, Cafes, Bars, Wine Bars, Re...
23                                  Restaurants, Italian
28     Cocktail Bars, Bars, Italian, Nightlife, Resta...
31                       Pizza, Restaurants, Salad, Soup
33                                    Pizza, Restaurants
41     Restaurants, Specialty Food, Steakhouses, Food...
45                                  Restaurants, Chinese
47     Coffee & Tea, Restaurants, Wine Bars, Bars, Ni...
53     Coffee & Tea, Cafes, Pets, Restaurants, Pet Ad...
59                                    Restaurants, Pizza
60                   Restaurants, Soup, Seafood, Burgers
61     Sports Bars, American (New), American (Traditi...
64     Seafood, Restaurants, Bars, Nightlife, Coc

## Logistic Regression with Unigram

#### Split into training and test sets

In [6]:
from sklearn.model_selection import train_test_split

print("Dataset size:", len(merged_data))
print(merged_data['cuisine_label'].value_counts())

x_input = merged_data['text']
y_output = merged_data['cuisine_label']

test_size = int(0.2 * len(merged_data))
x_train, x_test, y_train, y_test = train_test_split(x_input, y_output, test_size=test_size, random_state=9)
print(len(x_train), len(x_test))
print(len(y_train), len(y_test))

Dataset size: 4541
cuisine_label
American         2118
Italian           646
Mexican           625
Japanese          326
Chinese           303
Mediterranean     208
French            167
Vietnamese        102
Korean             46
Name: count, dtype: int64
3633 908
3633 908


## Extract Unigram Features

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

unigram_vectorizer = CountVectorizer()
unigram_vectorizer.fit(x_train)
train_features = unigram_vectorizer.transform(x_train)
test_features = unigram_vectorizer.transform(x_test)

print(train_features.shape) # prints (number of rows in the matrix, number of columns)
print(test_features.shape)  # prints (number of rows in the matrix, number of columns)

(3633, 13930)
(908, 13930)


## Train and Evaluate

In [8]:
from sklearn.linear_model import LogisticRegression

clf_unigrams = LogisticRegression(max_iter=1000) # Instantiate a logistic regression classifier
clf_unigrams.fit(train_features, y_train) # Train the classifier

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [9]:
# Evaluate unigram logistic regression classifier
from sklearn.metrics import classification_report # this provides a bunch of useful evaluation metrics

unigram_predictions = clf_unigrams.predict(test_features)

results = pd.DataFrame(classification_report(y_test, unigram_predictions, output_dict=True))
results

,American,Chinese,French,Italian,Japanese,Korean,Mediterranean,Mexican,Vietnamese,accuracy,macro avg,weighted avg
precision,0.681818,0.519231,0.750000,0.558333,0.851064,0.6,0.615385,0.650407,0.750000,0.660793,0.664026,0.661168
recall,0.835267,0.500000,0.225000,0.531746,0.740741,0.2,0.195122,0.645161,0.260870,0.660793,0.459323,0.660793
f1-score,0.750782,0.509434,0.346154,0.544715,0.792079,0.3,0.296296,0.647773,0.387097,0.660793,0.508259,0.641216
support,431.000000,54.000000,40.000000,126.000000,54.000000,15.0,41.000000,124.000000,23.000000,0.660793,908.000000,908.000000


## Formality Classification
Classifies restaurants as Casual, Classy, Elegant, etc. based on the business text.

In [10]:
# Uses this function to assign the formality level. Use both 'categories' and 'attributes'
# since both have keywords that apply to formality. 
def get_f(categories, attributes):
    categories = str(categories)
    attributes = str(attributes)

    # COMMENT: Shouldn't the unigram/n-gram model be able to figure out the keywords automatically?
    
    # Fine dining 
    # These keywords signify upscale dining experiences and taken from RestaurantsAttire
    if any(kw in categories for kw in ['Fine Dining', 'Steakhouses', 'French', 'Wine Bars', 'Sushi Bars', 'Brasseries', 'Italian']):
        return 'Fine Dining'
    if 'RestaurantsAttire' in attributes and 'formal' in attributes.lower():
        return 'Fine Dining'

    # Casual
    # These keywords show that it is quick service/informal dining taken also from the 
    # Restaurant Attire.
    if any(kw in categories for kw in ['Fast Food', 'Diners', 'Food Stands', 'Hot Dogs',
                                        'Cafes', 'Pizza', 'Burgers', 'Sandwiches']):
        return 'Casual'
    if 'RestaurantsAttire' in attributes and 'casual' in attributes.lower():
        return 'Casual'

    # Mid-range is the default for sit-down restaurants
    # Any remaining that did not seem applicable for upscale or casual are in this category.
    if 'Restaurants' in categories:
        return 'Mid-range'

    return None

In [11]:
print("Current business_data shape:", business_data.shape)
print("Sample categories:", business_data['categories'].head())
print("Sample attributes:", business_data['attributes'].head())

Current business_data shape: (2210, 15)
Sample categories: 3     Restaurants, Food, Bubble Tea, Coffee & Tea, B...
14          Food, Delis, Italian, Bakeries, Restaurants
15                    Sushi Bars, Restaurants, Japanese
19                                  Korean, Restaurants
20    Coffee & Tea, Food, Cafes, Bars, Wine Bars, Re...
Name: categories, dtype: str
Sample attributes: 3     {'RestaurantsDelivery': 'False', 'OutdoorSeati...
14    {'OutdoorSeating': 'False', 'RestaurantsGoodFo...
15    {'RestaurantsReservations': 'True', 'Restauran...
19    {'NoiseLevel': 'u'quiet'', 'GoodForMeal': '{'d...
20    {'OutdoorSeating': 'False', 'Caters': 'True', ...
Name: attributes, dtype: object


In [12]:
# Apply the formality labelling row-by-row

business_data['formality_label'] = business_data.apply(
    lambda row: get_f(row['categories'], row['attributes']), axis=1
)

# Merges the review text with formality labels on shared business_id

formality_merged = review_data.merge(
    business_data[['business_id', 'formality_label']], on='business_id'
)

print("After merge:", len(formality_merged))
print("Null labels:", formality_merged['formality_label'].isna().sum())

# Removed any non-restaurants
formality_merged = formality_merged.dropna(subset=['formality_label'])
print("After filtering:", len(formality_merged))
print(formality_merged['formality_label'].value_counts())

After merge: 7105
Null labels: 0
After filtering: 7105
formality_label
Casual         5367
Fine Dining    1564
Mid-range       174
Name: count, dtype: int64


## Training the Formality Model

In [13]:
# way 2
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Separate the review text and formality
x_f = formality_merged['text']
y_f = formality_merged['formality_label']
test_size_f = int(0.2 * len(formality_merged))
x_f_train, x_f_test, y_f_train, y_f_test = train_test_split(
    x_f, y_f, test_size=test_size_f, random_state=9
)

# Vectorize
vec_f = CountVectorizer()
vec_f.fit(x_f_train)
train_f = vec_f.transform(x_f_train)
test_f  = vec_f.transform(x_f_test)

# Train logistic regression on the unigram features
clf_formality = LogisticRegression(max_iter=1000)
clf_formality.fit(train_f, y_f_train)

# Predicts formality labels 
preds_f = clf_formality.predict(test_f)

# Display classification report with precision, recall, and F1 per class
results_f = pd.DataFrame(classification_report(y_f_test, preds_f, output_dict=True))
results_f

,Casual,Fine Dining,Mid-range,accuracy,macro avg,weighted avg
precision,0.833043,0.578544,0.500000,0.783955,0.637196,0.767542
recall,0.897844,0.477848,0.131579,0.783955,0.502424,0.783955
f1-score,0.864231,0.523397,0.208333,0.783955,0.531987,0.770897
support,1067.000000,316.000000,38.000000,0.783955,1421.000000,1421.000000


In [14]:
business_data['attributes'].apply(lambda x: x.get('RestaurantsAttire') if isinstance(x, dict) else None).value_counts()


attributes
u'casual'    1249
'casual'      766
u'dressy'      32
'dressy'       19
u'formal'       1
'formal'        1
Name: count, dtype: int64

## Zero-Shot LLM Cuisine Classification
Uses Gemma 3 (via Pitt's API gateway) to classify cuisine type directly from review text — no training required. Evaluated on a ~200-sample stratified subset of the test set.

In [22]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from sklearn.metrics import classification_report

load_dotenv()
client = OpenAI(
    api_key=os.environ["API_KEY"],
    base_url="https://ol.sci.pitt.edu"
)

CUISINES = ['Chinese', 'Italian', 'Mexican', 'French', 'Japanese', 'Korean', 'Mediterranean', 'Vietnamese', 'American']
CUISINE_LIST_STR = ', '.join(CUISINES)

def classify_cuisine_llm(review_text):
    prompt = (
        f"What cuisine type is this restaurant? "
        f"Reply with exactly one word from this list: {CUISINE_LIST_STR}. "
        f"Do not explain — just the single word.\n\n"
        f"Review: {review_text[:500]}"
    )
    response = client.chat.completions.create(
        model="gemma3",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10
    )
    pred = response.choices[0].message.content.strip()
    for cuisine in CUISINES:
        if cuisine.lower() in pred.lower():
            return cuisine
    return pred

# Stratified sample of ~200 from the test set, grouped directly on y_test
sample_idx = (
    y_test.groupby(y_test)
    .apply(lambda g: g.sample(min(len(g), max(1, int(200 * len(g) / len(y_test)))), random_state=42))
    .index.get_level_values(-1)
)
x_sample = x_test.loc[sample_idx]
y_sample = y_test.loc[sample_idx]

print(f"Classifying {len(x_sample)} reviews with Gemma 3 (this may take ~1-2 minutes)...")
llm_preds = [classify_cuisine_llm(text) for text in x_sample]

results_llm = pd.DataFrame(classification_report(y_sample, llm_preds, output_dict=True, zero_division=0))
results_llm

Classifying 195 reviews with Gemma 3 (this may take ~1-2 minutes)...


,American,Chinese,French,Indian,Italian,Japanese,Korean,Mediterranean,Mexican,Vietnamese,accuracy,macro avg,weighted avg
precision,0.710526,0.625000,1.00,0.0,0.727273,0.642857,1.000000,0.833333,0.761905,0.666667,0.712821,0.696756,0.732190
recall,0.861702,0.454545,0.25,0.0,0.592593,0.818182,0.333333,0.555556,0.592593,0.800000,0.712821,0.525850,0.712821
f1-score,0.778846,0.526316,0.40,0.0,0.653061,0.720000,0.500000,0.666667,0.666667,0.727273,0.712821,0.563883,0.702000
support,94.000000,11.000000,8.00,0.0,27.000000,11.000000,3.000000,9.000000,27.000000,5.000000,0.712821,195.000000,195.000000
